# Applied Statistics Project
## Population Decline In The West of Ireland
----
Population change is a primary indicator of regional development. It reflects long-term patterns in migration, employment opportunity and housing availability. In Ireland, the _Western Development Commsion_ was set up 
> "in response to public pressure to help tackle the massive population decline in the Western Region" - WesternDevelopment.ie

The focus of the [Western Development Commision's](https://westerndevelopment.ie/) remit - **Clare**, **Galway**, **Mayo**, **Roscommon**, **Sligo**, **Leitrim** and **Donegal**. As an area that I was born in to, emigrated from and returned to raise a family in, I have a vested interest in the region's success and seemed fitting to apply the analyitcal tools and methodologies to this topic. 

Thus far, throughout the course of the Applied Statistics Module, we have looked at -

    1. [Lady Tasting Tea Experiment](https://en.wikipedia.org/wiki/Lady_tasting_tea)

    2. [ANOVA (Analysis Of Variance)](https://en.wikipedia.org/wiki/Analysis_of_variance)

    3. [Normal Distribution](https://en.wikipedia.org/wiki/Normal_distribution)

    4. Permutations & Combinations

    5. [Binomial Distribution](https://numpy.org/doc/stable/reference/random/generated/numpy.random.binomial.html)

    6. Flipping Two & Many Coins

    7. Bell Curves

    8. [Histograms with Matplotlib](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hist.html)

    9. [Sampling Distribution](https://en.wikipedia.org/wiki/Sampling_distribution)

The aim of the project is to apply the methodologies from the above to the Central Statistics Office [_Population at Each Census_ dataset](https://data.cso.ie/table/F1001). I will quantitatively compare the population dynamics of the WDC counties with the rest of Ireland and assess whether the data reflects a different patterns of growth, decline and/or volatility. 



In [1]:
# Library Imports
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import json

In [ ]:
# Data Source
df = pd.read_csv('files/populationbycounty1841-2022.csv')

# Printing To Test
df.head()

,Statistic Label,CensusYear,County,Sex,UNIT,VALUE
0,Population at Each Census,1841,State,Both sexes,Number,6528799
1,Population at Each Census,1841,State,Male,Number,3222485
2,Population at Each Census,1841,State,Female,Number,3306314
3,Population at Each Census,1841,Carlow,Both sexes,Number,86228
4,Population at Each Census,1841,Carlow,Male,Number,42428


#### Pivotting The Data
Next step is to pivot this df so that each county has one row per year. The 'Male', 'Female' and 'Both sexes' will get their own column. The *UNIT* and *Statistic Label* columns will be dropped. I will be using the Pandas [*pivot_table*](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.pivot_table.html) function to perform this transformation. 

In [ ]:
census_pop = df.pivot_table(
    index=["CensusYear","County"], 
    columns="Sex",
    values="VALUE",
    aggfunc="sum"
).reset_index()

census_pop.columns.name=None
census_pop = census_pop.rename_axis(None, axis=1)

# Renaming Column
census_pop.rename(columns={"Both sexes": "FMCombined"}, inplace=True)

# Re-ordering Columns
census_pop = census_pop[["CensusYear", "County", "Female", "Male", "FMCombined"]]

# Printing To Test 
census_pop.head

<bound method NDFrame.head of      CensusYear     County  Female    Male  FMCombined
0          1841     Carlow   43800   42428       86228
1          1841      Cavan  122344  120814      243158
2          1841      Clare  142285  144109      286394
3          1841       Cork  433567  420551      854118
4          1841    Donegal  150627  145821      296448
..          ...        ...     ...     ...         ...
697        2022  Tipperary   84256   83639      167895
698        2022  Waterford   64268   63095      127363
699        2022  Westmeath   48500   47721       96221
700        2022    Wexford   83142   80777      163919
701        2022    Wicklow   79287   76564      155851

[702 rows x 5 columns]>

#### Grouping The Counties
I will add a new column to the dataframe to categorise the counties as either a **'WDC'** county or **'Rest'**. I will use the [*Lambda* function](https://www.w3schools.com/python/python_lambda.asp) to do this.

In [5]:
# Defining WDC Group
wdc = ["Clare", "Galway", "Mayo", "Roscommon", "Sligo", "Leitrim", "Donegal"]

# Adding Group Column With Lambda
census_pop["Group"] = census_pop["County"].apply(lambda x: "WDC" if x in wdc else ("State" if x == "State" else "Rest"))

# Creating Dataframes By Group
df_wdc = census_pop[census_pop["Group"] == "WDC"]
df_rest = census_pop[census_pop["Group"] == "Rest"]
df_state = census_pop[census_pop["Group"] == "State"]

,CensusYear,County,Female,Male,FMCombined,Group
21,1841,State,3306314,3222485,6528799,State
48,1851,State,2617079,2494478,5111557,State
75,1861,State,2233069,2169042,4402111,State
102,1871,State,2060719,1992468,4053187,State
129,1881,State,1957582,1912438,3870020,State
156,1891,State,1740093,1728601,3468694,State
183,1901,State,1611738,1610085,3221823,State
210,1911,State,1550179,1589509,3139688,State
237,1926,State,1465103,1506889,2971992,State
264,1936,State,1447966,1520454,2968420,State
